In [3]:
# QUESTION 1: Ridge Regression
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score

STUDENT_ID = "AIT2209938"
OUTDIR = f"./{STUDENT_ID}_quiz2_outputs"
os.makedirs(OUTDIR, exist_ok=True)

SALARY_PATH = r"C:\Users\Admin\Downloads\salary_prediction_dataset.csv"

try:
    df = pd.read_csv(SALARY_PATH)
    print(f"Dataset loaded successfully. Shape: {df.shape}")
except FileNotFoundError:
    print(f"ERROR: Could not find '{SALARY_PATH}'. Please check the file path.")
    raise

if 'Annual Salary' in df.columns and 'AnnualSalary' not in df.columns:
    df = df.rename(columns={'Annual Salary':'AnnualSalary'})

sample = df.sample(n=min(200, len(df)), random_state=42)
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()

if 'AnnualSalary' in df.columns:
    sns.pairplot(sample[num_cols])
    plt.suptitle('Pairplot (sample)', y=1.02)
    plt.savefig(os.path.join(OUTDIR, 'pairplot_sample.png'), bbox_inches='tight')
    plt.close()
    print("Saved pairplot_sample.png")

# Boxplot of AnnualSalary
plt.figure(figsize=(8,4))
sns.boxplot(x=df['AnnualSalary'])
plt.title('Boxplot of AnnualSalary')
plt.savefig(os.path.join(OUTDIR, 'boxplot_annualsalary.png'), bbox_inches='tight')
plt.close()
print("Saved boxplot_annualsalary.png")

# Correlation heatmap
plt.figure(figsize=(10,8))
sns.heatmap(df[num_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Correlation heatmap')
plt.savefig(os.path.join(OUTDIR, 'corr_heatmap.png'), bbox_inches='tight')
plt.close()
print("Saved corr_heatmap.png")

# Outliers via IQR
Q1 = df['AnnualSalary'].quantile(0.25)
Q3 = df['AnnualSalary'].quantile(0.75)
IQR = Q3 - Q1
outliers = df[(df['AnnualSalary'] < Q1 - 1.5*IQR) | (df['AnnualSalary'] > Q3 + 1.5*IQR)]
print('Num outliers (AnnualSalary):', len(outliers))


X = df.drop(columns=['AnnualSalary'])
y = df['AnnualSalary']


cat_cols = X.select_dtypes(include=['object','category']).columns.tolist()
if cat_cols:
    X = pd.get_dummies(X, drop_first=True)


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# Ridge regression
alphas = [0.001, 0.01, 0.1, 1, 10]
results = []
coefs = {}

for a in alphas:
    rr = Ridge(alpha=a, random_state=42)
    rr.fit(X_train_s, y_train)
    preds = rr.predict(X_test_s)
    
    rmse = mean_squared_error(y_test, preds, squared=False)
    r2 = r2_score(y_test, preds)
    
    results.append({'alpha':a, 'rmse':rmse, 'r2':r2})
    coefs[a] = pd.Series(rr.coef_, index=X.columns)

res_df = pd.DataFrame(results)
res_df.to_csv(os.path.join(OUTDIR,'ridge_results.csv'), index=False)

best = res_df.loc[res_df['rmse'].idxmin()]
best_alpha = best['alpha']
print(f'\nBest alpha (by RMSE): {best_alpha}')

# Save coefficients for best alpha
best_coefs = coefs[best_alpha].sort_values(key=lambda x: abs(x), ascending=False)
best_coefs.to_csv(os.path.join(OUTDIR,'best_alpha_coefficients.csv'))

with open(os.path.join(OUTDIR,'question1_report.txt'),'w') as f:
    f.write('Question 1 summary\n')
    f.write(res_df.to_string(index=False))
    f.write('\n\nTop coefficients (abs sorted):\n')
    f.write(best_coefs.to_string())

print(f'Question 1 outputs saved in: {OUTDIR}')

Dataset loaded successfully. Shape: (500, 9)
Saved pairplot_sample.png
Saved boxplot_annualsalary.png
Saved corr_heatmap.png
Num outliers (AnnualSalary): 0

Best alpha (by RMSE): 0.001
Question 1 outputs saved in: ./AIT2209938_quiz2_outputs


C:\Users\Admin\anaconda3\Lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
C:\Users\Admin\anaconda3\Lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
C:\Users\Admin\anaconda3\Lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
C:\Users\Admin\anaconda3\Lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the 